# Model testing

In [1]:
from fire_spread import fire_spread_model, clip_study_area, data_preparation_functions
import geopandas as gpd
import rasterio

In [2]:
# Load full data
fires_gpd = gpd.read_file("../01_Data/06_Wildfire_clusters/Fire_clusters_chaco.shp")
event_id = 5072  # Example event ID
year = fires_gpd[fires_gpd['CLUSTER_ID'] == event_id]['ACQ_DATE'].iloc[0].year

In [3]:
ca_data_full = data_preparation_functions.prepare_ca_inputs(
    event_id=event_id, 
    land_use_path=f'../01_Data/03_MapBiomas/{year}_coverage_lclu_25-1-1',
    weather_csv_path= '../01_Data/05_Spread_Covariates/02_Weather/era5_timeseries_5072.csv',
    srtm_path= '../01_Data/07_SRTM/SRTM_Paraguay_Chaco.tif',
    fire_points_gdf=fires_gpd)


Preparing CA inputs for event 5072

1. Loading elevation...
  Elevation loaded: (22256, 20369)
  Valid cells: 453332464
  NoData cells: 0
  Elevation range: 0.0 to 623.0 m

2. Finding ignition location...
  Ignition point (full grid): row=17330, col=15884
  Coordinates: x=-58.364500, y=-23.957400
  Ignition time: 2019-10-28 00:00:00

3. Clipping to 10 km buffer around ignition...
  Original grid: 22,256 × 20,369 = 453,332,464 cells (453.3 million)
  Clipped grid:  667 × 667 = 444,889 cells (0.44 million)
  Buffer: 10 km (333 cells)
  Memory reduction: 99.9%
  Ignition point (clipped grid): row=333, col=333

4. Calculating slope and aspect...
  Slope calculated: range 0.0 to 16.7 degrees
  Aspect calculated: range 0 to 6.22 radians

5. Loading land use...
  Land use loaded: (30849, 31147)
  Unique classes: 11

6. Clipping land use to same extent...
  Land use needs snapping before clipping...

7. Mapping to fuel types...
  Fuel type distribution:
    Type 1 (Ks=0.04): 81,072 cells (18.

In [36]:
# These will actually produce different results now!
print("Running CA simulation with Kr=1.0...")
result1 = fire_spread_model.run_ca_simulation(ca_data_full, Kr=1.0)
print("\nRunning CA simulation with Kr=5.0...")
result2 = fire_spread_model.run_ca_simulation(ca_data_full, Kr=5.0)
print("\nRunning CA simulation with Kr=10.0...")
result3 = fire_spread_model.run_ca_simulation(ca_data_full, Kr=7)

Running CA simulation with Kr=1.0...
t=0: Burning cells: 1, Time: 2019-10-28 00:00:00
  → New ignitions this timestep: 0
t=1: Burning cells: 0, Time: 2019-10-28 00:02:00
Fire extinguished at time step 1

Running CA simulation with Kr=5.0...
t=0: Burning cells: 1, Time: 2019-10-28 00:00:00
  → New ignitions this timestep: 1
t=1: Burning cells: 1, Time: 2019-10-28 00:02:00
  → New ignitions this timestep: 0
t=2: Burning cells: 0, Time: 2019-10-28 00:04:00
Fire extinguished at time step 2

Running CA simulation with Kr=10.0...
t=0: Burning cells: 1, Time: 2019-10-28 00:00:00
  → New ignitions this timestep: 2
t=1: Burning cells: 2, Time: 2019-10-28 00:02:00
  → New ignitions this timestep: 5
t=2: Burning cells: 4, Time: 2019-10-28 00:04:00
  → New ignitions this timestep: 3
t=3: Burning cells: 3, Time: 2019-10-28 00:06:00
  → New ignitions this timestep: 4
t=4: Burning cells: 4, Time: 2019-10-28 00:08:00
  → New ignitions this timestep: 4
t=5: Burning cells: 4, Time: 2019-10-28 00:10:00
 

In [ ]:
result2

{'burn_time': array([[-1, -1, -1, ..., -1, -1, -1],
        [-1, -1, -1, ..., -1, -1, -1],
        [-1, -1, -1, ..., -1, -1, -1],
        ...,
        [-1, -1, -1, ..., -1, -1, -1],
        [-1, -1, -1, ..., -1, -1, -1],
        [-1, -1, -1, ..., -1, -1, -1]], shape=(667, 667), dtype=int32),
 'final_state': array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], shape=(667, 667), dtype=int8),
 'time_step_minutes': np.float64(2.448724358260762),
 'total_burned_cells': np.int64(56)}

In [37]:
# data_preparation_functions.export_raster(result2['final_state'], 'Outputs/result2.tif', ca_data_full, dtype=rasterio.float32, nodata=None)
# data_preparation_functions.export_raster(result1['final_state'], 'Outputs/result1.tif', ca_data_full, dtype=rasterio.float32, nodata=None)
data_preparation_functions.export_raster(result3['final_state'], 'Outputs/result6.tif', ca_data_full, dtype=rasterio.float32, nodata=None)

  Exported: Outputs/result6.tif
